# Set H — BMI‑1: micro:bit Only  (LEGO–Colab)

**Final (Option 1 style)** — 2026-03-09\
**Author:** Satish Nair

Run locally if using serial (FEAR bridge). Includes systems‑theoretic callouts + Try with Copilot prompts.

---
## Table of Contents
- [H0 Starter](#h0)
- [H1 FEAR serial bridge](#h1)
- [H2 micro:bit firmware](#h2)
- [Reflection](#reflection)
---

## H0 — Starter (local runtime only) <a id='h0'></a>

In [ ]:

!pip -q install pyserial
import time, sys, serial, serial.tools.list_ports as list_ports
VID_MICROBIT=3368; PID_MICROBIT=516; BAUD=115200
_ser=None

def find_port():
    for p in list_ports.comports():
        if getattr(p,'vid',None)==VID_MICROBIT and getattr(p,'pid',None)==PID_MICROBIT:
            return p.device
    for p in list_ports.comports():
        d=(p.description or '').lower()
        if any(x in d for x in ['micro:bit','mbed','daplink','bbc']): return p.device
    return None

def open_serial():
    global _ser
    if _ser and _ser.is_open: return _ser
    port=find_port()
    if not port: raise RuntimeError('micro:bit not found')
    _ser=serial.Serial(port,BAUD,timeout=0.2); time.sleep(0.2); return _ser

def send_fear():
    s=open_serial(); s.write(b'FEAR
'); s.flush()

print('H0 ready.')


> **Systems callout — Local hardware loop**
- micro:bit accessed via USB serial; FEAR is a simple command protocol.
- Use only in local runtimes; Colab cannot access serial devices.

## H1 — FEAR activation logic (host side) <a id='h1'></a>

In [ ]:

import ipywidgets as w
from IPython.display import display

thr=w.FloatSlider(value=0.7,min=0,max=1,step=0.01,description='FEAR thr')
b=w.Button(description='Test FEAR',button_style='danger')

def on_fear(prob,thr=0.7,cd_ms=1500):
    if not hasattr(on_fear,'t0'): on_fear.t0=0
    now=time.time()*1000
    if prob>=thr and now-on_fear.t0>cd_ms:
        send_fear(); on_fear.t0=now; print('FEAR sent',prob)

def _t(_): send_fear()
b.on_click(_t)
display(w.VBox([thr,b]))


> **Systems callout — Thresholding & cooldown**
- FEAR triggered when probability crosses threshold and cooldown has elapsed.
- Prevents repeated triggers; implements a simple refractory mechanism analogous to spike‑based systems.

**Try with Copilot:**
> Integrate a real classifier output and choose thr/cooldown for stable behavior.

## H2 — micro:bit firmware (flash as main.py) <a id='h2'></a>

In [ ]:

from microbit import *
import music, uart
uart.init(baudrate=115200)

def fear():
    display.show(Image.SKULL)
    music.play(['c5:1','b4:1','a4:2'])
    display.clear()

while True:
    if uart.any():
        if uart.readline().strip().upper()==b'FEAR': fear()
    sleep(10)


> **Systems callout — Device‑side reflex**
- micro:bit listens for FEAR over UART; reacts with audio/visual signal.
- Acts as an actuator node in a larger BMI loop.

**Try with Copilot:**
> Add a second command (e.g., CALM) and design symmetric behaviors.

## Reflection <a id='reflection'></a>
- How would you replace FEAR with a multi‑command protocol?
- Where should debouncing/refractory logic live: host or device?
- How could you integrate ML models running in Colab with local hardware?